In [ ]:
# Linear Regression, the Mother Model
# Generated from the canonical HTML manuscript. Run this cell first.
# Source: https://github.com/Shakeri-Lab/dl-book/blob/c058d1f401fd0ead3ae59a2a8730f95489a2d9aa/chapters/part1/01-linear-regression.qmd

from importlib.metadata import PackageNotFoundError, version as package_version
import hashlib as _bootstrap_hashlib
import os as _bootstrap_os
from pathlib import Path as _BootstrapPath
import subprocess as _bootstrap_subprocess
import sys as _bootstrap_sys
import urllib.request as _bootstrap_urlrequest

_BOOK_REVISION = 'c058d1f401fd0ead3ae59a2a8730f95489a2d9aa'
_PINNED_REQUIREMENTS = [
    "torch==2.12.1",
    "torchvision==0.27.1",
    "numpy==2.5.1",
    "matplotlib==3.11.1"
]
_BOOK_ASSETS = []

def _installed_requirement(requirement: str) -> bool:
    name, expected = requirement.split('==', 1)
    try:
        return package_version(name) == expected
    except PackageNotFoundError:
        return False

_missing_requirements = [
    item for item in _PINNED_REQUIREMENTS if not _installed_requirement(item)
]
if _missing_requirements:
    _bootstrap_install = _bootstrap_subprocess.run(
        [_bootstrap_sys.executable, '-m', 'pip', 'install', '--quiet',
         *_missing_requirements],
        check=False, capture_output=True, text=True,
    )
    if _bootstrap_install.returncode != 0:
        raise RuntimeError(_bootstrap_install.stdout + _bootstrap_install.stderr)

_bootstrap_base = _BootstrapPath(
    _bootstrap_os.environ.get(
        'DLBOOK_NOTEBOOK_ROOT',
        '/content' if _BootstrapPath('/content').is_dir()
        else str(_BootstrapPath.home() / '.cache'),
    )
)
_BOOK_ROOT = _bootstrap_base / f'dl-book-{_BOOK_REVISION[:12]}'
_RAW_ROOT = 'https://raw.githubusercontent.com/Shakeri-Lab/dl-book/' + _BOOK_REVISION + '/'
for _record in _BOOK_ASSETS:
    _destination = _BOOK_ROOT / _record['path']
    _destination.parent.mkdir(parents=True, exist_ok=True)
    _valid = (
        _destination.is_file()
        and _bootstrap_hashlib.sha256(_destination.read_bytes()).hexdigest()
        == _record['sha256']
    )
    if not _valid:
        _temporary = _destination.with_suffix(_destination.suffix + '.part')
        _bootstrap_urlrequest.urlretrieve(_RAW_ROOT + _record['path'], _temporary)
        _digest = _bootstrap_hashlib.sha256(_temporary.read_bytes()).hexdigest()
        if _digest != _record['sha256']:
            _temporary.unlink(missing_ok=True)
            raise RuntimeError(f"Checksum mismatch for {_record['path']}")
        _temporary.replace(_destination)

(_BOOK_ROOT / 'chapters/part1').mkdir(parents=True, exist_ok=True)
_bootstrap_sys.path.insert(0, str(_BOOK_ROOT / 'code'))
_bootstrap_os.chdir(_BOOK_ROOT / 'chapters/part1')

# Hidden manuscript support required by later learner-visible cells.
# Plot-only harnesses are not exported.
import torch
from torch import nn

assert _BOOK_ROOT.is_dir()

**Plan**

1. Create a two-parameter regression problem and its loss surface.
2. Follow the gradient from a deliberately poor starting point.
3. Show the same landscape in perspective and from overhead.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# [1]
torch.manual_seed(6050)
x1 = torch.randn(60)
y1 = 2.5 * x1 - 1.0 + 0.3 * torch.randn(60)

def mse(w, b):
    return ((y1 - (w * x1 + b)) ** 2).mean()

W, B = np.meshgrid(
    np.linspace(-1.5, 5.5, 100),
    np.linspace(-4.0, 3.0, 100),
)
L = np.vectorize(lambda wi, bi: float(mse(wi, bi)))(W, B)

# [2]
w, b, path = -0.5, 2.0, []
for _ in range(20):
    path.append((w, b))
    err = (w * x1 + b) - y1
    w -= 0.25 * float(2 * (err * x1).mean())
    b -= 0.25 * float(2 * err.mean())

pw, pb = zip(*path)
path_loss = [float(mse(wi, bi)) for wi, bi in path]

# [3]
fig = plt.figure(figsize=(9.2, 3.8))
ax3d = fig.add_subplot(1, 2, 1, projection="3d")
ax3d.plot_surface(W, B, L, cmap="Blues", alpha=0.78, linewidth=0)
ax3d.plot(pw, pb, path_loss, "o-", color="#E57200", ms=3, lw=1.5)
ax3d.set(xlabel="$w$", ylabel="$b$", zlabel="MSE")
ax3d.view_init(elev=27, azim=-58)

ax2d = fig.add_subplot(1, 2, 2)
ax2d.contour(W, B, L, levels=25, cmap="Blues", alpha=0.85)
ax2d.plot(pw, pb, "o-", color="#E57200", ms=4, lw=1.5,
          label="descent path")
ax2d.plot(2.5, -1.0, "k*", ms=12, label="data-generating parameters")
ax2d.set(xlabel="$w$", ylabel="$b$")
ax2d.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Plan**

1. Load the chapter dependencies and establish reproducible state.

In [ ]:
import torch
import matplotlib.pyplot as plt

# [1]
torch.manual_seed(6050)

**Plan**

1. Define the reusable `make_synthetic_data` helper.
2. Generate noisy targets from known weights, then check the batch shapes.

In [ ]:
# [1]
def make_synthetic_data(
    weights: torch.Tensor, bias: float, n_samples: int, noise: float = 0.1
) -> tuple[torch.Tensor, torch.Tensor]:
    """y = Xw + b + noise.  Returns X: (n, d) and y: (n,)."""
    X = torch.randn(n_samples, len(weights))
    y = X @ weights + bias + noise * torch.randn(n_samples)
    return X, y

# [2]
true_w, true_b = torch.tensor([2.0, -3.4]), 4.2
X, y = make_synthetic_data(true_w, true_b, n_samples=200)
X.shape, y.shape        # always check your shapes

**Plan**

1. Append a constant feature to represent the bias.
2. Solve least squares with a rank-aware library routine.
3. Compare the estimate with the planted parameters.

In [ ]:
# [1]
X_aug = torch.cat([X, torch.ones(X.shape[0], 1)], dim=1)

# [2]
w_ols = torch.linalg.lstsq(X_aug, y).solution

# [3]
print(f"closed form:      w = {w_ols[:2].numpy().round(3)},  b = {w_ols[2]:.3f}")
print(f"ground truth:     w = {true_w.numpy()},  b = {true_b}")

**Plan**

1. Store the parameters and define the linear prediction.
2. Convert batch residuals into the exact MSE update.
3. Repeat that update until the loss settles.
4. Compare the learned parameters with the other solution.

In [ ]:
# [1]
class LinearRegressionScratch:
    """Linear regression trained with manually derived gradients."""

    def __init__(self, input_size: int, lr: float = 0.1):
        self.w = 0.01 * torch.randn(input_size)
        self.b = torch.zeros(1)
        self.lr = lr

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        return X @ self.w + self.b

    def step(self, X: torch.Tensor, y: torch.Tensor) -> float:
        # [2]
        err = self.forward(X) - y            # 1. predict, 2. measure   (n,)
        self.w -= self.lr * 2 * X.T @ err / len(y)   # 3. feel the slope,
        self.b -= self.lr * 2 * err.mean()           # 4. step downhill
        return float((err ** 2).mean())

# [3]
model = LinearRegressionScratch(input_size=2)
losses = [model.step(X, y) for _ in range(100)]

# [4]
print(f"gradient descent: w = {model.w.numpy().round(3)},  b = {model.b.item():.3f}")

**Plan**

1. Pair a linear module with MSE and an SGD optimizer.
2. Repeat the framework forward, backward, and update sequence.
3. Detach and report the learned parameters.

In [ ]:
from torch import nn

# [1]
net = nn.Linear(in_features=2, out_features=1)
optimizer = torch.optim.SGD(net.parameters(), lr=0.1)
loss_fn = nn.MSELoss()
nn_losses = []

# [2]
for _ in range(100):
    optimizer.zero_grad()              # clear old gradients: they accumulate!
    loss = loss_fn(net(X).squeeze(-1), y)   # (n, 1) -> (n)
    nn_losses.append(float(loss.detach()))
    loss.backward()                    # autograd feels the slope for us
    optimizer.step()

w_nn = net.weight.detach().squeeze()
b_nn = net.bias.item()
# [3]
print(f"nn.Linear:        w = {w_nn.numpy().round(3)},  b = {b_nn:.3f}")

**Plan**

1. Collect the direct-solve and iterative loss records.

In [ ]:
# [1]
closed_losses = [
    float((y ** 2).mean()),
    float(((X_aug @ w_ols - y) ** 2).mean()),
]
records = [closed_losses, losses, nn_losses]
titles = ["closed form", "scratch gradient descent", "PyTorch module"]

**Plan**

1. Choose a one-dimensional slice through the two-feature data.
2. Evaluate each learned model along that same slice.
3. Plot the adjusted observations and all three fits.

In [ ]:
# [1]
grid = torch.linspace(X[:, 0].min(), X[:, 0].max(), 50)
x2_mean = X[:, 1].mean()

# [2]
def prediction_slice(weights: torch.Tensor, bias: float) -> torch.Tensor:
    """Predictions along x1 while x2 is held at its sample mean."""
    return weights[0] * grid + weights[1] * x2_mean + bias

y_on_slice = y - true_w[1] * (X[:, 1] - x2_mean)

plt.figure(figsize=(5.5, 3.6))
# [3]
plt.scatter(X[:, 0], y_on_slice, s=12, alpha=0.4, label="data adjusted to slice")
plt.plot(grid, prediction_slice(w_ols, float(w_ols[2])), lw=3, label="closed form")
plt.plot(grid, prediction_slice(model.w, model.b.item()), "--", lw=2,
         label="gradient descent")
plt.plot(grid, prediction_slice(w_nn, b_nn), ":", lw=2, label="nn.Linear")
plt.xlabel("$x_1$"); plt.ylabel("$y$"); plt.legend()
plt.tight_layout()
plt.show()

**Plan**

1. Create a small-sample problem with many irrelevant features.
2. Solve the same data with unregularized and ridge estimators.
3. Compare both estimates with the planted weights.

In [ ]:
# [1]
n, d = 25, 20                            # barely more samples than knobs!
w_true = torch.zeros(d)
w_true[:5] = torch.tensor([3.0, -2.0, 1.5, 2.5, -1.0])   # only 5 real features
Xh, yh = make_synthetic_data(w_true, bias=0.0, n_samples=n, noise=2.0)

# [2]
w_ols_h = torch.linalg.lstsq(Xh, yh).solution
lam = 0.4
w_ridge = torch.linalg.solve(Xh.T @ Xh + n * lam * torch.eye(d), Xh.T @ yh)

# [3]
def report(name: str, w_hat: torch.Tensor) -> None:
    err = ((w_hat - w_true) ** 2).mean()
    print(f"{name:>6}:  weight MSE = {err:.4f}")

report("OLS", w_ols_h)
report("ridge", w_ridge)